# 01 Load raw Municipal Money data into Lakehouse tables
Upload the six Parquet files from `data/raw/` into the Lakehouse under **Files/raw/** first, then run all cells.
Each file becomes a Delta table with the prefix `raw_`. No cleaning happens here: cleaning waits for the missingness decisions.

In [ ]:
import re  # tools for finding and replacing patterns in text (used to fix column names)
from notebookutils import mssparkutils  # Fabric's helper library, used here to list files in the Lakehouse

files = [f for f in mssparkutils.fs.ls("Files/raw") if f.name.endswith(".parquet")]  # list everything in the Files/raw folder and keep only the Parquet files
print([f.name for f in files])  # show the file names we found (should be 6)

In [ ]:
def clean_cols(df):
    """Make column names safe for Delta tables: Delta doesn't allow dots or spaces."""
    for c in df.columns:  # go through every column name
        df = df.withColumnRenamed(c, re.sub(r"[^0-9a-zA-Z_]", "_", c))  # replace anything that isn't a letter, digit or underscore with _ (demarcation.code -> demarcation_code)
    return df  # hand back the table with the new names

for f in files:  # go through each Parquet file
    name = "raw_" + f.name.replace(".parquet", "")  # build the table name: audit_opinions.parquet -> raw_audit_opinions
    df = clean_cols(spark.read.parquet(f.path))  # read the file into a Spark table and fix its column names
    df.write.mode("overwrite").format("delta").saveAsTable(name)  # save it in the Lakehouse as a Delta table, replacing any older copy
    print(f"{name}: {df.count():,} rows, {len(df.columns)} columns")  # report how many rows and columns were saved

In [ ]:
%%sql
-- quick check: count the audit opinions in each year, by type of opinion
SELECT financial_year_end_year AS year,  -- the financial year, renamed to "year"
       opinion_code,                     -- the type of audit opinion (e.g. qualified)
       COUNT(*) AS n                     -- how many municipalities got that opinion that year
FROM raw_audit_opinions                  -- from the audit opinions table we just loaded
GROUP BY financial_year_end_year, opinion_code  -- one row per year and opinion type
ORDER BY year, opinion_code              -- sorted by year, then opinion